# Optional Vision 02 - What survives the smaller code?

The CNN can call a bar "vertical" while forgetting exactly where it was.
Rebuilding the image needs that lost detail. A two-score decision is no
longer enough.

**Guiding question:** How much visual evidence can a compact code preserve?

Keep local 16×16 grayscale images, now adding diagonals and square rings
beside vertical/horizontal bars. No pretrained weights or downloads are
needed.

| Complaint | What we will inspect |
|---|---|
| Direct copying teaches no compact representation | Force the image through a smaller number grid |
| Matching output shape can still rebuild badly | Train against matching pixels; compare with all-black |
| Average error hides missing strokes | Pair originals with reconstructions on one brightness scale |
| "Compressed" is too vague | Count values per image and per code |
| More room may not earn its cost | Matched-budget two/four-channel trials |

We'll train an encoder/decoder and inspect its actual outputs and code.
This isn't a variational autoencoder or a test that random codes generate
valid shapes. The branch remains optional for language models.
[Handwritten theory](02-pytorch-autoencoders-theory.md).


## Change the checking answer, not every familiar idea

An **encoder** packs pixels into an intermediate **latent code**.
A **decoder** uses that code to rebuild pixels. Reconstruction error
teaches both ends; a species or orientation label isn't required.

The generator creates 480 noisy images. The first 400 are training,
the last 80 validation. Four shape labels remain as audit tags for
comparing codes later, but the model receives only images and the loss
checks only images.

`[480,1,16,16]` means examples, one grayscale channel, rows, columns.
Each image has 256 pixel values. Merely returning those pixels unchanged
would solve reconstruction without forcing any smaller representation;
the next architecture closes that shortcut.


In [ ]:
import sys
print(f"Kernel interpreter: {sys.executable}")

# -- Setup and local image dataset ------------------------------------------------
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

torch.manual_seed(21)
np.random.seed(21)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white"})

def make_image(index, size=16):
    image = torch.zeros(size, size)
    label = index % 4
    offset = int(torch.randint(-2, 3, ()).item())
    if label == 0:
        image[3:13, 7 + offset:9 + offset] = 1
    elif label == 1:
        image[7 + offset:9 + offset, 3:13] = 1
    elif label == 2:
        coordinates = torch.arange(3, 13)
        image[coordinates, coordinates + offset] = 1
    else:
        image[4:12, 4:12] = 1
        image[6:10, 6:10] = 0
    image += 0.05 * torch.randn_like(image)
    return image.clamp(0, 1), label

examples = [make_image(index) for index in range(480)]
# 480 separate (16, 16) pixel grids -> (480, 16, 16) -> (480, 1, 16, 16).
# Stack makes image rows; the singleton grayscale channel supplies Conv2d's channel axis.
images = torch.stack([item[0] for item in examples]).unsqueeze(1)
labels = torch.tensor([item[1] for item in examples])
train_images, valid_images = images[:400], images[400:]
train_labels, valid_labels = labels[:400], labels[400:]
loader = DataLoader(TensorDataset(train_images, train_labels), batch_size=40, shuffle=True)
print(f"Local images: {tuple(images.shape)}")


## 1 - Make every image fit through the smaller opening

![An image passes through an encoder into a four-channel four-by-four code, then a decoder rebuilds it; matching original and rebuilt pixels supply the loss.](../../images/vision-rebuild-through-a-code.png)

Read the class by following one batch:

```text
pixels [B,1,16,16]
-> stride-two convolution [B,8,8,8]
-> stride-two convolution [B,L,4,4]      the code
-> learned expansion [B,8,8,8]
-> learned expansion [B,1,16,16]        rebuilt pixels
```

`L=4` means four channels at sixteen positions: **64 code values**, not
four class scores. The code is an activation produced for an image;
the encoder and decoder weights are the stored trainable parameters.

The decoder's transposed convolutions learn upsampling. They aren't an
inverse that can recover anything the encoder threw away.
Sigmoid bounds pixels to `[0,1]`, but an in-range value can still be wrong.

> **Visual to add — Pack 256 pixel slips into 64 code slots**
> Show one 16×16 square-ring image with 256 pixel positions entering a
> four-layer stack of 4×4 code slots, 64 total, then a rebuilt 16×16 grid.
> Keep the ring's hollow center as the detail to preserve; don't label
> code channels as shape names or invent a perfect reconstruction.
> Suggested caption: "The decoder must rebuild from what fits through the code."

The next cell returns **both** reconstruction and latent activation for
five images. Its equality assertion checks output geometry only. We
haven't taught the packer or unpacker anything yet.


In [ ]:
# -- Encoder, decoder, and explicit shape contract -------------------------------
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_channels=4):
        super().__init__()
        # (B images, 1 channel, 16, 16) -> (B, 8, 8, 8) -> (B, L, 4, 4).
        # Stride two shrinks each spatial axis; L=latent_channels controls code size.
        # ReLU keeps each shape while adding a nonlinear feature transform.
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(8, latent_channels, 3, stride=2, padding=1),
            nn.ReLU(),
        )
        # (B, L, 4, 4) -> (B, 8, 8, 8) -> (B, 1, 16, 16): learned
        # upsampling restores the input pixel layout so reconstruction loss can compare
        # matching coordinates. Sigmoid keeps that layout and bounds values to [0, 1].
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(latent_channels, 8, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(8, 1, 4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, inputs):
        latent = self.encoder(inputs)
        return self.decoder(latent), latent

model = ConvAutoencoder(latent_channels=4)
reconstruction, latent = model(train_images[:5])
print(f"Input:          {tuple(train_images[:5].shape)}")
print(f"Latent:         {tuple(latent.shape)}")
print(f"Reconstruction: {tuple(reconstruction.shape)}")
assert reconstruction.shape == train_images[:5].shape


### A mostly dark picture makes a lazy baseline look respectable

Most pixels are background. An all-black answer can have a small average
error while omitting every shape. Measure that baseline before celebrating
the autoencoder's MSE.

The next loop takes batches of 40 images and ignores the loader's shape
labels. It predicts pixels, compares each with its matching input pixel,
clears gradients, backpropagates, and lets Adam change both modules.

MSE averages across examples, channels, rows, and columns. All ten
training batches have size 40, so averaging their means is fair here.
`history` mixes losses measured as the model changes within each epoch;
validation uses one fixed final state.

> **Visual to add — A low pixel average can lose the whole stroke**
> Put the same diagonal or square-ring validation image above two
> comparison panels: all-black baseline and "model reconstruction—use
> measured output later". Highlight that many background pixels can match
> while foreground detail is absent. Keep brightness `[0,1]`; don't attach
> invented MSEs. Suggested caption: "Background agreement isn't shape recovery."

The block checks that training error fell and **prints** whether validation
beat black. It doesn't assert a winner that the run might not deliver.


In [ ]:
# -- Train reconstruction from pixels --------------------------------------------
# (80, 1, 16, 16) pixel errors -> scalar mean: can a blank reconstruction already look good?
black_baseline_mse = nn.functional.mse_loss(torch.zeros_like(valid_images), valid_images).item()
print(f"All-black validation baseline MSE: {black_baseline_mse:.6f}")
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
history = []
for epoch in range(35):
    epoch_losses = []
    for batch, _ in loader:
        reconstruction, _ = model(batch)
        loss = nn.functional.mse_loss(reconstruction, batch)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        epoch_losses.append(float(loss.detach()))
    # Ten equally sized batch means -> scalar epoch mean; no remainder bias here.
    history.append(np.mean(epoch_losses))

with torch.inference_mode():
    valid_reconstruction, valid_latent = model(valid_images)
    valid_mse = nn.functional.mse_loss(valid_reconstruction, valid_images).item()
print(f"Training MSE: {history[0]:.6f} -> {history[-1]:.6f}")
print(f"Validation MSE: {valid_mse:.6f}")
print("Beats all-black baseline:", valid_mse < black_baseline_mse)
assert history[-1] < history[0]


### Give the mean some pictures to answer to

The next gallery puts eight actual validation originals above their
reconstructions with the same `[0,1]` brightness range. Automatic contrast
could make a faint, wrong image look pleasantly sharp.

Look for location, orientation, thickness, and the ring's open center.
The per-image MSE then averages within each image, leaving 80 separate
scores. The first eight printed values correspond to the eight columns.

A low mean can hide a missing diagonal; a recognizable shape can still
have substantial pixel error. Neither the plot nor one scalar replaces
the other. No training occurs during this inspection.

Inference mode skips gradient tracking. These convolution, ReLU, and sigmoid
layers have no train/eval difference today; adding dropout or batch normalization
would also require explicit evaluation mode, as in Part 05.


In [ ]:
# -- Original versus reconstructed evidence --------------------------------------
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for index in range(8):
    axes[0, index].imshow(valid_images[index, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, index].imshow(valid_reconstruction[index, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, index].axis("off")
    axes[1, index].axis("off")
axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Rebuilt")
plt.show()
# Pixel misses (80, 1, 16, 16) -> (80,): average within each image, not across examples.
per_image_mse = (valid_reconstruction - valid_images).square().mean(dim=(1, 2, 3))
print("First eight reconstruction MSEs:", per_image_mse[:8].tolist())


## 2 - Count the code before naming its meaning

The input holds `1×16×16=256` values per image.
The four-channel code holds `4×4×4=64`: a **4:1 value-count ratio**.
That doesn't include precision, model weights, headers, or an actual
compressed-file format.

The next block flattens `[80,4,4,4]` into `[80,64]` without mixing images.
It then uses the four shape labels **only to group the audit**, averages
each group's codes, and returns a `[4,4]` table of distances between
group centers.

Separated centers don't prove that one channel means "diagonal", that
every individual example separates, or that any coordinate has a stable
human name. Useful information may be distributed across many coordinates.
Read the measured distance table as that smaller inspection.


In [ ]:
# -- Latent-space inspection and compression rate --------------------------------
# Non-batch shape tuples -> scalar products: count values for ONE image/code.
input_values = int(np.prod(valid_images.shape[1:]))
latent_values = int(np.prod(valid_latent.shape[1:]))
compression_ratio = input_values / latent_values
class_centers = []
# (80 images, 4 channels, 4, 4) -> (80, 64): keep one row per image,
# joining its code coordinates so distances compare whole codes, not separate patches.
flattened_latent = valid_latent.flatten(1)
for label in range(4):
    # (20 audit images, 64 coordinates) -> (64,): average examples, preserve code coordinates.
    class_centers.append(flattened_latent[valid_labels == label].mean(dim=0))
# Four mean codes (64,) -> (4 classes, 64 coordinates); cdist compares all
# row pairs -> (4, 4), one class-pair distance per cell, not extra image examples.
center_distances = torch.cdist(torch.stack(class_centers), torch.stack(class_centers))

print(f"Input values per image: {input_values}")
print(f"Latent values per image: {latent_values}")
print(f"Compression ratio: {compression_ratio:.1f}x")
print("Distances between class-average latent codes:")
print(center_distances.round(decimals=2))


## 3 - Does extra room earn its cost?

![Two-channel and four-channel codes hold thirty-two and sixty-four values respectively; compare both compression and measured reconstruction quality rather than assuming a winner.](../../images/vision-code-budget.png)

Two latent channels hold `2×4×4=32` values: an **8:1** value-count ratio.
Four hold 64: **4:1**. Both still decode to a 16×16 image.

The next helper trains a fresh model for **18 epochs per width**, resetting
the same seed recipe, using the same rows, loader, loss, and Adam rate.
It returns validation MSE plus value-count ratio. Width changes parameter
shapes, so fairness can't mean identical weight arrays.

The main model trained for 35 epochs. Don't compare its score with an
18-epoch trial and attribute the difference to width.
**Predict:** Must the four-channel trial win on MSE? Read both measured
scores and both exact counts. Add width 8 only after the default pair is
clear; it holds 128 values, a 2:1 ratio.

Once these validation scores guide a width choice, a final quality claim
would need fresh evidence beyond this small generator split.


In [ ]:
# -- Your turn: one controlled latent-width comparison ---------------------------
def train_width(latent_channels, epochs=18):
    torch.manual_seed(21)
    candidate = ConvAutoencoder(latent_channels)
    optimizer = torch.optim.Adam(candidate.parameters(), lr=0.01)
    for _ in range(epochs):
        for batch, _ in loader:
            rebuilt, _ = candidate(batch)
            loss = nn.functional.mse_loss(rebuilt, batch)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
    with torch.inference_mode():
        rebuilt, latent_code = candidate(valid_images)
    # Reconstruction errors -> scalar MSE; latent shape product -> values per image.
    return (
        nn.functional.mse_loss(rebuilt, valid_images).item(),
        input_values / int(np.prod(latent_code.shape[1:])),
    )

# CHANGE THIS: add 8 to compare a wider latent code
widths = [2, 4]
results = {width: train_width(width) for width in widths}
for width, (mse, ratio) in results.items():
    print(f"latent channels={width}: validation MSE={mse:.6f}, compression={ratio:.1f}x")
print("Check both reconstruction error and compression; neither metric owns the decision alone.")


## The short version

- A class label can discard detail that reconstruction needs. Here the
  original image supplies the target; shape labels only organize an audit.
- The code is an input-dependent activation. Encoder/decoder parameters
  are the stored rules that make and read it.
- Learned expansion restores shape, not information guaranteed to survive.
  Matching `[B,1,16,16]` doesn't mean matching pixels.
- Sparse backgrounds make the black baseline important. Use measured
  validation MSE, paired pictures, and per-image errors together.
- Count per image: 256 pixels become 64 values at width 4, or 32 at
  width 2. These are value ratios, not compressed-file sizes.
- Group-center distances aren't named concepts or a separability test.
- The width trials share an 18-epoch budget, not the main model's 35.

| Coverage | Where we stopped |
|---|---|
| Built and measured | Convolutional encoder/decoder; pixel training; black baseline; validation reconstructions; latent counts; group centers; matched-width trials |
| Explained | Parameters versus codes; learned upsampling; count versus byte compression; reconstruction versus classification |
| Later | A VAE needs a distributional objective and sampling tests; random-code generation, real-image compression, and a new final test aren't run here |


**Next complaint:** rebuilding an image from its own code does not teach a model to predict the next item in an ordered stream. Return to [Sequence Memory](../../../03-language-model-foundations/01-sequence-memory/README.md) for that language-route question. The familiar `nn.Module`, named axes, loss, and update loop carry forward; the image dataset and reconstruction objective do not.